# RoPE and attention

(Notebook companion to `scripts/rope_demo.py`. Self-contained: only external libraries.)

**Attention** lets every token look at earlier tokens and mix in what it finds. It works by comparing a
*query* from the current token with a *key* from each earlier token (a dot product = a score), turning the
scores into weights with softmax, and taking a weighted sum of the *values*.

On its own, that comparison is blind to word order: it only sees what the tokens are, not where they are.
**RoPE** (rotary position embeddings) fixes this by *rotating* the query and key vectors by an angle that
depends on their position, just before the dot product. The result is that the score between two tokens
depends on **how far apart they are**.

Where it sits in each attention layer:

    x ──► q_proj ──► q ──► RoPE ─┐
      ├─► k_proj ──► k ──► RoPE ─┼─► q·kᵀ / √d ─► causal mask ─► softmax ─► · v ─► o_proj
      └─► v_proj ──► v  (no RoPE)┘

RoPE touches **q and k only**, never v.

Select the kernel **smalllm (.venv)** before running.

In [1]:
import math

import torch
import torch.nn.functional as F
from torch import Tensor, nn

torch.manual_seed(0)
torch.set_printoptions(precision=3, linewidth=140, sci_mode=False)

HEAD_DIM = 64
ROPE_THETA = 100000.0

## 1. The RoPE functions
Three small functions. `rope_cos_sin` builds the angles for each position, `rotate_half` builds the
"quarter-turned" copy of a vector, and `apply_rope` combines them: `x * cos + rotate_half(x) * sin`.

In [2]:
def rope_cos_sin(positions: Tensor, head_dim: int, theta: float) -> tuple[Tensor, Tensor]:
    """cos/sin tables for integer positions, shape (T, head_dim)."""
    inv_freq = 1.0 / theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)
    freqs = positions.float()[:, None] * inv_freq[None, :]  # angle = position * frequency
    emb = torch.cat((freqs, freqs), dim=-1)  # dims i and i + head_dim/2 share one angle
    return emb.cos(), emb.sin()


def rotate_half(x: Tensor) -> Tensor:
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x: Tensor, cos: Tensor, sin: Tensor) -> Tensor:
    """x: (B, n_heads, T, head_dim); cos/sin: (T, head_dim)."""
    return x * cos + rotate_half(x) * sin

## 2. Each pair of dimensions rotates at its own speed
A head vector of 64 numbers is treated as 32 pairs. Pair `i` turns by `position × frequency_i` radians. The
frequencies run from fast to very slow, so fast pairs tell neighbouring tokens apart and slow pairs track
long distances (like the hands of a clock).

In [3]:
inv_freq = 1.0 / ROPE_THETA ** (torch.arange(0, HEAD_DIM, 2, dtype=torch.float32) / HEAD_DIM)
print(f"{HEAD_DIM // 2} pairs. Radians per position, fastest 3: {inv_freq[:3].tolist()}")
print(f"slowest 3: {[f'{v:.2e}' for v in inv_freq[-3:].tolist()]}")
print(f"fastest pair repeats every {2 * math.pi / inv_freq[0].item():.1f} tokens")
print(f"slowest pair repeats every {2 * math.pi / inv_freq[-1].item():,.0f} tokens")

32 pairs. Radians per position, fastest 3: [1.0, 0.6978305578231812, 0.486967533826828]
slowest 3: ['2.94e-05', '2.05e-05', '1.43e-05']
fastest pair repeats every 6.3 tokens
slowest pair repeats every 438,460 tokens


## 3. Which dimensions are paired
Hugging Face's Llama layout pairs dimension `i` with `i + head_dim/2`: first half with second half. It is
**not** neighbouring (even/odd) pairs. Published SmolLM2 weights were trained with this layout, so using the
other one would silently break them.

In [4]:
x = torch.arange(8, dtype=torch.float32)
print("x              =", x.tolist())
print("rotate_half(x) =", rotate_half(x).tolist())
print("dim 0 is paired with dim 4, dim 1 with dim 5, and so on.")

x              = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
rotate_half(x) = [-4.0, -5.0, -6.0, -7.0, 0.0, 1.0, 2.0, 3.0]
dim 0 is paired with dim 4, dim 1 with dim 5, and so on.


## 4. RoPE is a 2D rotation of each pair
Take pair 0 (dimensions 0 and 32). Its frequency is 1.0, so at position 5 it should be rotated by 5 radians.
We check `apply_rope` against an ordinary rotation done by hand. A rotation never changes a vector's length.

In [5]:
x = torch.randn(1, 1, 1, HEAD_DIM)
pos = 5
cos, sin = rope_cos_sin(torch.tensor([pos]), HEAD_DIM, ROPE_THETA)
y = apply_rope(x, cos, sin)

angle = pos * 1.0
a, b = x[0, 0, 0, 0].item(), x[0, 0, 0, HEAD_DIM // 2].item()
by_hand = (a * math.cos(angle) - b * math.sin(angle), a * math.sin(angle) + b * math.cos(angle))
print(f"pair 0 before:          ({a:.4f}, {b:.4f})")
print(f"pair 0 after RoPE:      ({y[0, 0, 0, 0].item():.4f}, {y[0, 0, 0, HEAD_DIM // 2].item():.4f})")
print(f"pair 0 rotated by hand: ({by_hand[0]:.4f}, {by_hand[1]:.4f})")
print(f"vector length before / after: {x.norm().item():.4f} / {y.norm().item():.4f}")

pair 0 before:          (-1.1258, -0.6136)
pair 0 after RoPE:      (-0.9077, 0.9055)
pair 0 rotated by hand: (-0.9077, 0.9055)
vector length before / after: 8.3705 / 8.3705


## 5. The payoff: scores depend only on distance
Put the *same* query vector at every position and the *same* key vector at every position, rotate both with
RoPE, and compute all query-key scores. Entry `[m, n]` is the score of the query at position `m` against the
key at position `n`. Because rotations by `m` and `n` combine into a rotation by `m - n`, every diagonal is
constant: only the distance matters.

In [6]:
T = 6
q = torch.randn(HEAD_DIM)
k = torch.randn(HEAD_DIM)
cos, sin = rope_cos_sin(torch.arange(T), HEAD_DIM, ROPE_THETA)
qs = apply_rope(q.expand(1, 1, T, HEAD_DIM), cos, sin)[0, 0]
ks = apply_rope(k.expand(1, 1, T, HEAD_DIM), cos, sin)[0, 0]
scores = qs @ ks.T
print(scores)
print("score(m=2, n=1) == score(m=5, n=4):", torch.allclose(scores[2, 1], scores[5, 4], atol=1e-4))

tensor([[ 5.542,  4.569,  2.754,  2.001,  3.089,  5.073],
        [ 4.401,  5.542,  4.569,  2.754,  2.001,  3.089],
        [ 1.811,  4.401,  5.542,  4.569,  2.754,  2.001],
        [-0.035,  1.811,  4.401,  5.542,  4.569,  2.754],
        [ 0.729, -0.035,  1.811,  4.401,  5.542,  4.569],
        [ 3.957,  0.729, -0.035,  1.811,  4.401,  5.542]])
score(m=2, n=1) == score(m=5, n=4): True


## 6. An attention layer
The same attention layer as the model's: grouped-query attention (several query heads share each key/value
head), no biases, RoPE on q and k, and a causal mask so a token cannot see later tokens. Here we build it
with a small size (hidden 128, 2 query heads, 1 key/value head).

In [7]:
HIDDEN, N_HEADS, N_KV_HEADS = 128, 2, 1


class Attention(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.q_proj = nn.Linear(HIDDEN, N_HEADS * HEAD_DIM, bias=False)
        self.k_proj = nn.Linear(HIDDEN, N_KV_HEADS * HEAD_DIM, bias=False)
        self.v_proj = nn.Linear(HIDDEN, N_KV_HEADS * HEAD_DIM, bias=False)
        self.o_proj = nn.Linear(N_HEADS * HEAD_DIM, HIDDEN, bias=False)

    def forward(self, x: Tensor, cos: Tensor, sin: Tensor) -> Tensor:
        b, t, _ = x.shape
        q = self.q_proj(x).view(b, t, N_HEADS, HEAD_DIM).transpose(1, 2)
        k = self.k_proj(x).view(b, t, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
        v = self.v_proj(x).view(b, t, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)  # RoPE on q and k only
        n_rep = N_HEADS // N_KV_HEADS  # each kv head serves n_rep consecutive query heads
        k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(b, t, -1))


attn = Attention()
for name, p in attn.named_parameters():
    nn.init.normal_(p, std=0.02)  # the model's init
    print(f"{name:14} {tuple(p.shape)}")

q_proj.weight  (128, 128)
k_proj.weight  (64, 128)
v_proj.weight  (64, 128)
o_proj.weight  (128, 128)


## 7. Same token five times: only RoPE can tell the positions apart
We feed attention the **same token at five positions**, so every token's content is identical. Without RoPE
the queries and keys are all the same, so every allowed token gets the same weight: attention cannot know
word order. With RoPE the weights change with distance.

(Freshly initialised weights are tiny, so all scores would be about 0 and softmax flat either way. We scale
q and k up by `BOOST`, as training eventually would, to make the effect visible.)

In [8]:
BOOST = 6.0
T = 5
x = torch.randn(1, 1, HIDDEN).expand(1, T, HIDDEN)  # one vector repeated at 5 positions

q = attn.q_proj(x).view(1, T, N_HEADS, HEAD_DIM).transpose(1, 2) * BOOST
k = attn.k_proj(x).view(1, T, N_KV_HEADS, HEAD_DIM).transpose(1, 2) * BOOST
cos, sin = rope_cos_sin(torch.arange(T), HEAD_DIM, ROPE_THETA)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))


def attention_weights(qq: Tensor, kk: Tensor) -> Tensor:
    raw = (qq[0, 0] @ kk[0, 0].T) / math.sqrt(HEAD_DIM)  # head 0
    return raw.masked_fill(~mask, float("-inf")).softmax(-1)


print("WITHOUT RoPE (rows = query position, columns = key position):")
print(attention_weights(q, k))
print("\nWITH RoPE:")
print(attention_weights(apply_rope(q, cos, sin), apply_rope(k, cos, sin)))

WITHOUT RoPE (rows = query position, columns = key position):
tensor([[1.000, 0.000, 0.000, 0.000, 0.000],
        [0.500, 0.500, 0.000, 0.000, 0.000],
        [0.333, 0.333, 0.333, 0.000, 0.000],
        [0.250, 0.250, 0.250, 0.250, 0.000],
        [0.200, 0.200, 0.200, 0.200, 0.200]], grad_fn=<SoftmaxBackward0>)

WITH RoPE:
tensor([[1.000, 0.000, 0.000, 0.000, 0.000],
        [0.698, 0.302, 0.000, 0.000, 0.000],
        [0.657, 0.239, 0.103, 0.000, 0.000],
        [0.539, 0.303, 0.110, 0.048, 0.000],
        [0.331, 0.360, 0.203, 0.074, 0.032]], grad_fn=<SoftmaxBackward0>)


Without RoPE every row is flat. With RoPE the weights depend on distance: this query attends more to nearby
keys here, and training can shape that into rules like "look at the previous token" or "look
three tokens back". (The raw scores repeat along diagonals, as in section 5; softmax then rescales each row so the weights sum to 1.)

## 8. The full computation by hand
Every step of attention written out, with RoPE applied to q and k and not to v, matching the module's output.

In [9]:
T = 5
x = torch.randn(1, T, HIDDEN)
cos, sin = rope_cos_sin(torch.arange(T), HEAD_DIM, ROPE_THETA)

# 1. Project to queries, keys, values and split into heads.
q = attn.q_proj(x).view(1, T, N_HEADS, HEAD_DIM).transpose(1, 2)
k = attn.k_proj(x).view(1, T, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
v = attn.v_proj(x).view(1, T, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
print("q, k, v shapes:", tuple(q.shape), tuple(k.shape), tuple(v.shape))

# 2. Rotate q and k by position. v is untouched.
q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)

# 3. Share each kv head with its group of query heads.
k, v = k.repeat_interleave(N_HEADS // N_KV_HEADS, dim=1), v.repeat_interleave(N_HEADS // N_KV_HEADS, dim=1)

# 4. Scores, causal mask, softmax, weighted sum of values.
scores = (q @ k.transpose(-1, -2)) / math.sqrt(HEAD_DIM)
scores = scores.masked_fill(~torch.tril(torch.ones(T, T, dtype=torch.bool)), float("-inf"))
weights = scores.softmax(-1)
mixed = (weights @ v).transpose(1, 2).reshape(1, T, -1)

# 5. Output projection.
by_hand = attn.o_proj(mixed)
print("max |by hand - Attention module| =", (by_hand - attn(x, cos, sin)).abs().max().item())

q, k, v shapes: (1, 2, 5, 64) (1, 1, 5, 64) (1, 1, 5, 64)
max |by hand - Attention module| = 9.313225746154785e-09


## 9. Position is only given to attention
Nothing else in the model knows about position: the embedding, the norms and the MLP treat each token
independently. RoPE inside attention is the **only** place order enters, and because it is a rotation of q
and k rather than something added to the embedding, no position table is stored and nothing extra lands in
the weights file. The same code also works for any sequence length up to what the model was trained on.